<a href="https://colab.research.google.com/github/aravindpunyamantula/ai-mentor-portfolio/blob/main/Day2_Lab2B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-genai pydantic
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [ ]:
from pydantic import BaseModel
from typing import List, Optional
class Education(BaseModel):
  degree: str
  institution: str
  year: int
class Resume(BaseModel):
  name: str
  email: str
  phone: Optional[str] = None
  education: List[Education]
  skills: List[str]
  projects: List[str] = []
  experience_years: float

In [ ]:
from google import genai
from pydantic import ValidationError
import os

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            # Retry once with the broken JSON in the prompt
            fix_prompt = (f'Fix this JSON to match schema. Errors: {e}. '
                          f'Original: {resp.text}')
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)

In [10]:
!pip install -q pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 1.6 MB/s eta 0:00:00


In [11]:
import pypdf
with open('/content/resume_new.pdf', 'rb') as f:
    reader = pypdf.PdfReader(f)
    page = reader.pages[0]
    resumes = [page.extract_text().strip()]

print(f'Loaded {len(resumes)} sample résumés')

results = []
for i, r in enumerate(resumes[:3]):
    try:
        parsed = extract_resume(r)
        results.append(parsed)
        print(f'\nRésumé {i+1}: {parsed.name} — {len(parsed.skills)} skills, '
              f'{parsed.experience_years} years exp')
    except Exception as e:
        print(f'\nRésumé {i+1}: FAILED — {type(e).__name__}: {str(e)[:200]}')

# Print full first result
if results:
    print('\n=== Full first result ===')
    print(results[0].model_dump_json(indent=2))

Loaded 1 sample résumés

Résumé 1: Punyamantula Durga Santosh Aravind Kumar — 41 skills, 1.0 years exp

=== Full first result ===
{
  "name": "Punyamantula Durga Santosh Aravind Kumar",
  "email": "aravindpunyamantula630@gmail.com",
  "phone": "+91-6304114648",
  "education": [
    {
      "degree": "B.Tech in Information Technology",
      "institution": "Aditya College of Engineering and Technology",
      "year": 2023
    },
    {
      "degree": "Intermediate",
      "institution": "Manasa Junior College (MPC) – Defence College",
      "year": 2023
    }
  ],
  "skills": [
    "TypeScript",
    "JavaScript (ES6+)",
    "Java",
    "Dart",
    "Python",
    "HTML",
    "CSS",
    "React.js",
    "React Hooks",
    "Tailwind CSS",
    "Flutter",
    "Firebase (Auth, Firestore, Storage)",
    "Razorpay SDK",
    "Node.js",
    "Express.js",
    "Fastify",
    "REST APIs",
    "JWT Authentication",
    "Microservices",
    "Docker",
    "Docker Compose",
    "MongoDB",
    "PostgreSQL"

In [13]:
try:
    bad = extract_resume('')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print('Caught gracefully:', type(e).__name__)
    print('Message:', str(e)[:200])

Unexpected success: {"name":"John Doe","email":"john.doe@example.com","phone":"555-123-4567","education":[{"degree":"Master of Science in Computer Science","institution":"University of Example","year":2022},{"degree":"Bachelor of Science in Software Engineering","institution":"Another University","year":2020}],"skills":["Python","Java","C++","JavaScript","React","SQL","AWS","Docker","Machine Learning"],"projects":["E-commerce Platform Development","AI-powered Recommendation System"],"experience_years":3.5}


## Day 2 Lab 2B — Errors handled
1. **Markdown fence wrapping** (`\`\`\`json ... \`\`\``)
The retry prompt asks Gemini to output raw JSON without fences. Triggers on ~5-10% of calls.
2. **Hallucinated phone number when source has none**
`Optional[str] = None` in Pydantic — model returns `null`, schema validates.
3. **Empty / whitespace-only input**
Pydantic raises ValidationError with "Field required". Caller catches.
## Sample résumés processed: 3 / 3 successful

```
# This is formatted as code
```

